In [ ]:
#conda create -n tei-nlp python=3.10
#conda activate tei-nlp
#                                                                               
# To deactivate an active environment, use                                      
#                                                                               
#     $ conda deactivate 
#pip install torch sentence-transformers lxml numpy
pip install jupyter ipykernel
python -m ipykernel install --user --name tei-nlp --display-name "TEI NLP"

In [ ]:
from lxml import etree
from sentence_transformers import SentenceTransformer, util
import numpy as np

# Load TEI
tree = etree.parse("/Users/gcrane/github/GRC_misc/heike.sadler1918.xml")
root = tree.getroot()

# Extract tokens (very basic; can adjust for your TEI structure)
tokens = root.xpath("//text()")  # or use //w for tokenized texts
print('tokens found',len(tokens))



In [ ]:
# Load embedding model
model = SentenceTransformer('all-mpnet-base-v2')
print('model loaded')
# Emotion concept descriptions
emotion_prompts = {
    "sadness": "a feeling of sorrow, grief, or deep unhappiness",
    "anger": "a feeling of rage, hostility, or fury",
    "joy": "a feeling of happiness, delight, or pleasure",
    "fear": "a feeling of dread, anxiety, or terror",
    "desire": "a feeling of longing or wanting something strongly"
}

emotion_embs = {
    e: model.encode(desc, convert_to_tensor=True)
    for e, desc in emotion_prompts.items()
}

results = []

for token in tokens:
    tok_emb = model.encode(token, convert_to_tensor=True)
    # Compute similarity to all emotions
    sims = {e: float(util.cos_sim(tok_emb, emb)) for e, emb in emotion_embs.items()}
    # Choose best
    best_emotion = max(sims, key=sims.get)
    if sims[best_emotion] > 0.45:   # empirical threshold
        results.append((token, best_emotion, sims[best_emotion]))